# AutoEIT — Automated EIT Transcription Pipeline
### GSoC 2026 Evaluation | HumanAI Foundation (NIU × University of Alabama)

---

## What This Notebook Does

The **Elicited Imitation Task (EIT)** measures Spanish L2 oral proficiency. Participants listen to 30 sentences of increasing complexity and repeat them back. This notebook automates the transcription of those responses from raw audio recordings.

### Pipeline Overview

```
4 × raw .mp3 files  (038010, 038011, 038012, 038015)
         │
         ▼
  ┌─────────────────────────────────────────────────────┐
  │  STAGE 1 — WhisperX + Speaker Diarization           │
  │  • Separates the two voices (experimenter + learner) │
  │  • Transcribes with L2-aware initial prompt          │
  │  • Writes output/<id>/transcript.txt per file        │
  └───────────────────────┬─────────────────────────────┘
                          │
                          ▼
  ┌─────────────────────────────────────────────────────┐
  │  STAGE 2 — Dynamic Sentence Alignment               │
  │  • Finds English/Spanish section boundary            │
  │  • Fuzzy-matches each segment to 1 of 30 stimuli    │
  │  • Handles restarts, merges, and missed responses    │
  │  • Writes [no response] for missing sentences        │
  │  • Fills column C of the Excel template             │
  └─────────────────────────────────────────────────────┘
         │
         ▼
  AutoEIT_Sample_Audio_for_Transcribing_FILLED.xlsx
```

### Results Summary

| Participant | File | Responses | Notes |
|:---:|---|:---:|---|
| 038010 | EIT-2A | 29/30 | Sentence 19: no response |
| 038011 | EIT-1A | 30/30 | |
| 038012 | EIT-2A | 28/30 | Very low-proficiency participant; 2 sentences unintelligible |
| 038015 | EIT-1A | 30/30 | |

**117 / 120 sentences filled across 4 participants.**

---

### Key Audio Structure Detail

Each recording has this layout — critical for correct processing:

```
[0s  – ~74s]    Silence
[~74s – ~121s]  ⚠️  English practice section  ← MUST BE SKIPPED
[~121s – ~154s] Long silence (~32s) = English/Spanish boundary
[~154s – end]   30 Spanish EIT trials
                  Each trial: [stimulus plays] → [tone] → [learner responds]
```

The pipeline auto-detects the boundary using language detection + silence gap analysis.

---
## Section 1 — Setup

**Runtime:** Change to T4 GPU before running → *Runtime → Change runtime type → T4 GPU*

**HuggingFace token:** Required for pyannote diarization model. [Accept the license here](https://huggingface.co/pyannote/speaker-diarization-3.1), then add your token to Colab Secrets as `HF_TOKEN`.

In [2]:
# # ── 1A: Install dependencies ──────────────────────────────────────────────────
# !pip install whisperx pydub -q
# !pip install rapidfuzz openpyxl langdetect -q
# !apt-get install -y ffmpeg -q
# !pip install torch torchvision torchaudio --upgrade -q
# print("✅ Dependencies installed")

In [4]:
# ── 1B: Create folder structure ───────────────────────────────────────────────
!mkdir -p ./input ./output

# Upload your 4 mp3 files to ./input/
# Upload AutoEIT_Sample_Audio_for_Transcribing.xlsx to ./  (root)
print("Folders ready. Upload your files:")
print("  ./input/   → 038010_EIT-2A.mp3, 038011_EIT-1A.mp3, 038012_EIT-2A.mp3, 038015_EIT-1A.mp3")
print("  ./         → AutoEIT_Sample_Audio_for_Transcribing.xlsx")

Folders ready. Upload your files:
  ./input/   → 038010_EIT-2A.mp3, 038011_EIT-1A.mp3, 038012_EIT-2A.mp3, 038015_EIT-1A.mp3
  ./         → AutoEIT_Sample_Audio_for_Transcribing.xlsx


In [5]:
# ── 1C: Verify uploads ────────────────────────────────────────────────────────
import os
from pathlib import Path

mp3_files = sorted(Path("./input").glob("*.mp3"))
excel     = Path("./AutoEIT Sample Audio for Transcribing.xlsx")

print(f"MP3 files found: {len(mp3_files)}")
for f in mp3_files:
    print(f"  ✅  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

print()
print(f"Excel template: {'✅' if excel.exists() else '❌ MISSING'}  {excel}")

MP3 files found: 4
  ✅  038010_EIT-2A.mp3  (9.3 MB)
  ✅  038011_EIT-1A.mp3  (9.6 MB)
  ✅  038012_EIT-2A.mp3  (19.0 MB)
  ✅  038015_EIT-1A.mp3  (9.0 MB)

Excel template: ✅  AutoEIT Sample Audio for Transcribing.xlsx


---
## Section 2 — Compatibility Patches

WhisperX and pyannote have dependency conflicts with newer versions of PyTorch and HuggingFace Hub. These patches fix them at runtime without requiring downgrading any packages. **Run this cell before any WhisperX import.**

In [6]:
import warnings
warnings.filterwarnings("ignore")

import torch
import torchaudio
import huggingface_hub
import soundfile as sf

# ── Patch 1: PyTorch 2.6+ weights_only strict loading ─────────────────────────
if not getattr(torch, '_load_is_patched', False):
    _orig_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._load_is_patched = True

# ── Patch 2: HuggingFace Hub deprecated keyword arguments ─────────────────────
_orig_hf_hub_download = huggingface_hub.hf_hub_download
def _patched_hf_hub_download(*args, **kwargs):
    if 'use_auth_token' in kwargs: kwargs['token'] = kwargs.pop('use_auth_token')
    kwargs.pop('resume_download', None)
    kwargs.pop('force_filename', None)
    kwargs.pop('local_dir_use_symlinks', None)
    try:
        return _orig_hf_hub_download(*args, **kwargs)
    except Exception as e:
        if "custom.py" in str(kwargs.get('filename', '')) and (
                "404" in str(e) or "EntryNotFound" in type(e).__name__):
            raise ValueError("File not found")
        raise e
huggingface_hub.hf_hub_download = _patched_hf_hub_download

_orig_model_info = huggingface_hub.model_info
def _patched_model_info(*args, **kwargs):
    if 'use_auth_token' in kwargs: kwargs['token'] = kwargs.pop('use_auth_token')
    return _orig_model_info(*args, **kwargs)
huggingface_hub.model_info = _patched_model_info

# ── Patch 3: torchaudio API compatibility ─────────────────────────────────────
class DummyAudioMetaData:
    def __init__(self, sr, frames, ch):
        self.sample_rate = sr; self.num_frames = frames; self.num_channels = ch

def _patched_info(uri, **kwargs):
    info = sf.info(uri)
    return DummyAudioMetaData(info.samplerate, info.frames, info.channels)

def _patched_load(uri, **kwargs):
    data, sr = sf.read(uri, always_2d=True, dtype='float32')
    return torch.from_numpy(data).t(), sr

def _patched_save(filepath, src, sample_rate, **kwargs):
    sf.write(filepath, src.t().numpy(), sample_rate)

if not hasattr(torchaudio, 'AudioMetaData'):        torchaudio.AudioMetaData = DummyAudioMetaData
if not hasattr(torchaudio, 'list_audio_backends'):  torchaudio.list_audio_backends = lambda: ['soundfile', 'sox_io']
if not hasattr(torchaudio, 'info'):                 torchaudio.info = _patched_info
if not hasattr(torchaudio, 'load'):                 torchaudio.load = _patched_load
if not hasattr(torchaudio, 'save'):                 torchaudio.save = _patched_save

print(f"✅ Patches applied")
print(f"   PyTorch  : {torch.__version__}")
print(f"   CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU      : {torch.cuda.get_device_name(0)}")

✅ Patches applied
   PyTorch  : 2.10.0+cu128
   CUDA     : True
   GPU      : Tesla T4


---
## Section 3 — HuggingFace Token

The diarization model (pyannote) requires a HuggingFace token. Store it as `HF_TOKEN` in Colab Secrets (*🔑 key icon in left sidebar*) — never paste it directly into the notebook.

In [7]:
def load_hf_token():
    """Load HF token: Colab secrets → environment variable → hardcoded fallback."""
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            print(f"  ✅ HF token loaded from Colab secrets: {token[:8]}...")
            return token
    except Exception:
        pass

    token = os.environ.get('HF_TOKEN', '')
    if token:
        print(f"  ✅ HF token from environment: {token[:8]}...")
        return token

    hardcoded = ""   # ← paste token here only as absolute last resort
    if hardcoded:
        return hardcoded

    print("  ❌ HF_TOKEN not found. Add it to Colab Secrets as 'HF_TOKEN'.")
    print("     Then accept the pyannote model license at:")
    print("     https://huggingface.co/pyannote/speaker-diarization-3.1")
    return ""

import os
HF_TOKEN = load_hf_token()

  ✅ HF token loaded from Colab secrets: hf_JUpIv...


---
## Section 4 — Stage 1: Speaker Separation & Transcription

**WhisperX** transcribes the audio and **pyannote** diarizes it to separate the two speakers (experimenter and learner). Models are loaded once and reused for all 4 files.

**Expected runtime:** ~8–12 min per file on T4 GPU → ~40 min total for all 4.

In [8]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
WHISPER_MODEL = "large-v3"   # best accuracy; use "medium" on CPU
TARGET_DBFS   = -20.0        # normalise audio to this level
LANGUAGE      = "es"         # force Spanish; prevents language detection errors

# Whisper initial prompt — encodes the full EIT transcription protocol
INITIAL_PROMPT = (
    "Un estudiante de español L2 repite una frase. "
    "Transcribe exactamente lo que dice. "
    "Incluye errores gramaticales, disfluencias (eh, mm, um), "
    "falsos inicios con guión (ej: la- las), y pausas con '...'. "
    "Si hay una pausa demasiado larga, probablemente indica el inicio de la siguiente oración. "
    "No corrijas ni normalices el texto. "
    "No traduzcas ninguna frase en español al inglés."
)

print(f"Model       : {WHISPER_MODEL}")
print(f"Language    : {LANGUAGE}")
print(f"Prompt len  : {len(INITIAL_PROMPT)} chars")

Model       : large-v3
Language    : es
Prompt len  : 373 chars


In [9]:
import whisperx
from whisperx.diarize import DiarizationPipeline
from pydub import AudioSegment
from collections import defaultdict

device  = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"Device: {device.upper()}  |  Compute: {compute}")

# ── Load models once for all files ────────────────────────────────────────────
print("\nLoading WhisperX model…")
asr_options   = {"initial_prompt": INITIAL_PROMPT}
whisper_model = whisperx.load_model(WHISPER_MODEL, device, compute_type=compute,
                                    asr_options=asr_options)
print("  ✅ Whisper ready")

print("Loading alignment model…")
align_model, align_meta = whisperx.load_align_model(language_code=LANGUAGE, device=device)
align_meta["language"]  = LANGUAGE
print("  ✅ Alignment model ready")

print("Loading diarization pipeline…")
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
print("  ✅ Diarization pipeline ready")

Device: CUDA  |  Compute: float16

Loading WhisperX model…


vocabulary.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

embedding/pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

  ✅ Diarization pipeline ready


In [10]:
def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m:02d}:{s:02d}"


def process_file(mp3_path, file_output_dir):
    """
    Run WhisperX + diarization on one mp3.
    Writes transcript.txt and speaker_1.mp3 / speaker_2.mp3 to file_output_dir.
    Returns number of segments.
    """
    os.makedirs(file_output_dir, exist_ok=True)
    wav_path = os.path.join(file_output_dir, "_working.wav")

    # Load + normalize audio
    audio_pydub = AudioSegment.from_mp3(mp3_path)
    total_s     = len(audio_pydub) / 1000
    delta       = TARGET_DBFS - audio_pydub.dBFS
    audio_pydub = audio_pydub.apply_gain(delta)
    audio_pydub.export(wav_path, format="wav")
    audio_arr   = whisperx.load_audio(wav_path)
    print(f"  Loaded  {total_s:.0f}s  ({delta:+.1f} dB normalisation)")

    # Transcribe
    result   = whisper_model.transcribe(audio_arr, batch_size=16, language=LANGUAGE)
    language = result.get("language", LANGUAGE)
    print(f"  Transcription done  (language: {language})")

    # Align
    result = whisperx.align(result["segments"], align_model, align_meta, audio_arr, device)
    print("  Alignment done")

    # Diarise
    diarize_segs = diarize_model(audio_arr, min_speakers=2, max_speakers=2)
    result       = whisperx.assign_word_speakers(diarize_segs, result)
    print("  Diarisation done")

    # Build clean segment list
    raw_speakers = sorted({seg.get("speaker","UNKNOWN")
                           for seg in result["segments"] if seg.get("speaker")})
    spk_map = {raw: f"Speaker {i+1}" for i, raw in enumerate(raw_speakers)}

    segments = []
    for seg in result["segments"]:
        raw = seg.get("speaker", "UNKNOWN")
        segments.append({
            "speaker":  spk_map.get(raw, "Unknown"),
            "start":    seg["start"],  "end":      seg["end"],
            "start_ms": int(seg["start"]*1000), "end_ms": int(seg["end"]*1000),
            "text":     seg["text"].strip(),
        })

    # Write transcript.txt
    transcript_path = os.path.join(file_output_dir, "transcript.txt")
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write("=" * 70 + "\n")
        f.write(f"  TRANSCRIPT — {os.path.basename(mp3_path)}\n")
        f.write("=" * 70 + "\n")
        current_spk = None
        for seg in segments:
            if seg["speaker"] != current_spk:
                current_spk = seg["speaker"]
                f.write(f"\n{'─'*70}\n{current_spk}:\n")
            f.write(f"  [{format_time(seg['start'])} → {format_time(seg['end'])}]  {seg['text']}\n")
        # Stats footer
        stats = defaultdict(lambda: {"count":0,"duration":0.0})
        for seg in segments:
            stats[seg["speaker"]]["count"]    += 1
            stats[seg["speaker"]]["duration"] += seg["end"] - seg["start"]
        f.write(f"\n{'='*70}\nSPEAKER STATS\n{'='*70}\n")
        for spk, st in sorted(stats.items()):
            f.write(f"  {spk}: {st['count']} segments, {st['duration']:.1f}s "
                    f"({st['duration']/total_s*100:.1f}%)\n")

    # Save per-speaker audio
    spk_audio = {}
    for seg in segments:
        spk   = seg["speaker"]
        chunk = audio_pydub[seg["start_ms"]:seg["end_ms"]]
        spk_audio[spk] = spk_audio.get(spk, AudioSegment.silent(0)) + chunk
    for spk, clip in sorted(spk_audio.items()):
        fname = f"{spk.replace(' ','_').lower()}.mp3"
        clip.export(os.path.join(file_output_dir, fname), format="mp3")

    os.remove(wav_path)
    print(f"  ✅ {len(segments)} segments  →  {transcript_path}")
    return len(segments)

In [11]:
# ── Run Stage 1 on all 4 files ────────────────────────────────────────────────
from pathlib import Path

mp3_files = sorted(Path("./input").glob("*.mp3"))
if not mp3_files:
    print("❌ No MP3 files found in ./input/")
else:
    stage1_summary = []
    for i, mp3_path in enumerate(mp3_files, 1):
        pid         = mp3_path.stem
        output_dir  = f"./output/{pid}"
        print(f"\n{'═'*60}")
        print(f"  FILE {i}/{len(mp3_files)}  —  {mp3_path.name}")
        print(f"{'═'*60}")
        try:
            n = process_file(str(mp3_path), output_dir)
            stage1_summary.append((mp3_path.name, "✅", n))
        except Exception as e:
            print(f"  ❌ Failed: {e}")
            stage1_summary.append((mp3_path.name, "❌", str(e)))

    print(f"\n{'═'*60}")
    print("  STAGE 1 COMPLETE")
    print(f"{'═'*60}")
    for fname, status, val in stage1_summary:
        info = f"{val} segments" if isinstance(val, int) else val
        print(f"  {status}  {fname:<35}  {info}")


════════════════════════════════════════════════════════════
  FILE 1/4  —  038010_EIT-2A.mp3
════════════════════════════════════════════════════════════
  Loaded  550s  (+26.8 dB normalisation)
  Transcription done  (language: es)
  Alignment done
  Diarisation done
  ✅ 52 segments  →  ./output/038010_EIT-2A/transcript.txt

════════════════════════════════════════════════════════════
  FILE 2/4  —  038011_EIT-1A.mp3
════════════════════════════════════════════════════════════
  Loaded  547s  (+17.2 dB normalisation)
  Transcription done  (language: es)
  Alignment done
  Diarisation done
  ✅ 57 segments  →  ./output/038011_EIT-1A/transcript.txt

════════════════════════════════════════════════════════════
  FILE 3/4  —  038012_EIT-2A.mp3
════════════════════════════════════════════════════════════
  Loaded  1126s  (+23.2 dB normalisation)
  Transcription done  (language: es)
  Alignment done
  Diarisation done
  ✅ 87 segments  →  ./output/038012_EIT-2A/transcript.txt

══════════════

---
## Section 5 — Stage 2: Sentence Alignment & Excel Write

Reads each `transcript.txt`, finds the English/Spanish boundary, then uses fuzzy matching with anchor gravity to map every transcribed segment to one of the 30 known EIT stimuli. Missing sentences receive `[no response]`.

In [13]:
import re

try:
    from langdetect import detect as _langdetect
    _LANGDETECT_OK = True
except ImportError:
    _LANGDETECT_OK = False

# ── The 30 EIT Version A stimuli ──────────────────────────────────────────────
EIT_STIMULI = [
    "Quiero cortarme el pelo",                                        # 01  7 syl
    "El libro está en la mesa",                                       # 02  7
    "El carro lo tiene Pedro",                                        # 03  8
    "Él se ducha cada mañana",                                        # 04  9
    "¿Qué dice usted que va a hacer hoy?",                            # 05  9
    "Dudo que sepa manejar muy bien",                                 # 06  10
    "Las calles de esta ciudad son muy anchas",                       # 07  11
    "Puede que llueva mañana todo el día",                            # 08  12
    "Las casas son muy bonitas pero caras",                           # 09  12
    "Me gustan las películas que acaban bien",                        # 10  12
    "El chico con el que yo salgo es español",                        # 11  13
    "Después de cenar me fui a dormir tranquilo",                     # 12  13
    "Quiero una casa en la que vivan mis animales",                   # 13  14
    "A nosotros nos fascinan las fiestas grandiosas",                 # 14  14
    "Ella sólo bebe cerveza y no come nada",                          # 15  15
    "Me gustaría que el precio de las casas bajara",                  # 16  15
    "Cruza a la derecha y después sigue todo recto",                  # 17  15
    "Ella ha terminado de pintar su apartamento",                     # 18  15
    "Me gustaría que empezara a hacer más calor pronto",              # 19  15
    "El niño al que se le murió el gato está triste",                 # 20  16
    "Una amiga mía cuida a los niños de mi vecino",                   # 21  16
    "El gato que era negro fue perseguido por el perro",              # 22  16
    "Antes de poder salir él tiene que limpiar su cuarto",            # 23  16
    "La cantidad de personas que fuman ha disminuido",                # 24  17
    "Después de llegar a casa del trabajo tomé la cena",              # 25  17
    "El ladrón al que atrapó la policía era famoso",                  # 26  17
    "Le pedí a un amigo que me ayudara con la tarea",                 # 27  17
    "El examen no fue tan difícil como me habían dicho",              # 28  17
    "¿Serías tan amable de darme el libro que está en la mesa?",      # 29  18
    "Hay mucha gente que no toma nada para el desayuno",              # 30  18
]

# ── Phrases that are NEVER participant responses ──────────────────────────────
SKIP_PHRASES = [
    # EIT protocol instructions
    "repite lo máximo que puedas", "repite lo más que puedas",
    "recuerda no comiences", "no comiences a repetir",
    "darás suficiente tiempo", "hasta que escuches",
    "ahora comenzamos", "vamos a comenzar", "presta atención",
    "no tomes notas", "no anotes nada",
    "now let s begin", "now let us begin", "let s begin",
    # Whisper prompt hallucination fragments
    "incluye errores gramaticales",
    "transcribe exactamente lo que dice",
    "falsos inicios con guion", "falsos inicios con guión",
    "no corrijas ni normalices", "no traduzcas ninguna frase",
    "si hay una pausa demasiado larga",
    "probablemente indica el inicio de la siguiente",
    "un estudiante de español l2 repite",
]

ENGLISH_MARKERS = {
    "the","you","your","will","please","after","each","until","start",
    "hear","tone","sentence","repeat","notes","recording","attention",
    "bought","drive","drove","park","call","buy","meat","butcher",
    "brother","computer","sometimes","volleyball","gym",
}

def normalize(text):
    text = text.lower()
    text = re.sub(r"[¿¡.,!?;:\"'«»\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def fuzzy(a, b):
    from rapidfuzz import fuzz
    return fuzz.token_sort_ratio(normalize(a), normalize(b))

def is_english(text):
    if len(text.split()) < 2: return False
    if _LANGDETECT_OK:
        try: return _langdetect(text) == "en"
        except: pass
    return len(set(normalize(text).split()) & ENGLISH_MARKERS) >= 2

def should_skip(text):
    if is_english(text): return True
    n = normalize(text)
    return any(p in n for p in SKIP_PHRASES)

print(f"✅ {len(EIT_STIMULI)} stimuli loaded  |  {len(SKIP_PHRASES)} skip phrases")

✅ 30 stimuli loaded  |  23 skip phrases


In [14]:
def parse_transcript(path):
    """Parse transcript.txt → list of {start, end, text} dicts."""
    segs, pattern = [], re.compile(r"\[(\d{2}:\d{2})\s*[→\-]+\s*(\d{2}:\d{2})\]\s*(.*)")
    def to_sec(t):
        mm, ss = t.split(":"); return int(mm)*60+int(ss)
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = pattern.search(line.strip())
            if not m: continue
            text = re.sub(r"^(Speaker\s*\d+|Unknown)\s+","",m.group(3),flags=re.IGNORECASE).strip()
            if text:
                segs.append({"start":to_sec(m.group(1)),"end":to_sec(m.group(2)),"text":text})
    return segs


def find_boundary(segs):
    """
    Return index of first Spanish segment.
    Priority: (1) last English/instruction segment in first 35% of audio,
              (2) largest silence gap in first 35%, (3) absolute last English.
    """
    if len(segs) < 2: return 0
    limit = segs[-1]["end"] * 0.35

    last_en = -1
    for i, s in enumerate(segs):
        if s["start"] > limit: break
        if is_english(s["text"]) or should_skip(s["text"]): last_en = i
    if last_en >= 0:
        print(f"    Boundary after seg #{last_en}  ({segs[last_en]['start']}s: '{segs[last_en]['text'][:50]}')")
        return last_en + 1

    best_gap, best_i = 0, 0
    for i in range(1, len(segs)):
        if segs[i-1]["start"] < 60 or segs[i-1]["start"] > limit: continue
        gap = segs[i]["start"] - segs[i-1]["end"]
        if gap > best_gap: best_gap, best_i = gap, i
    if best_i:
        print(f"    Boundary at seg #{best_i} via silence gap ({best_gap:.0f}s)")
        return best_i

    last_en = max((i for i,s in enumerate(segs) if is_english(s["text"]) or should_skip(s["text"])), default=-1)
    if last_en >= 0: return last_en + 1
    return 0


def split_merged(text):
    """Try to split one segment that contains two merged responses."""
    # Split on '...' first
    if "..." in text:
        parts = text.split("...")
        for cut in range(1, len(parts)):
            left  = "...".join(parts[:cut]).strip().rstrip(".")
            right = "...".join(parts[cut:]).strip().lstrip(".")
            if len(left.split()) >= 2 and len(right.split()) >= 2:
                _, li, ls = best_match(left)
                _, ri, rs = best_match(right)
                _, _, ss  = best_match(text)
                if li != ri and ls >= 40 and rs >= 40 and (ls+rs) > ss+15:
                    return [left, right]
    # Brute-force word boundary
    words = text.split()
    if len(words) < 8: return [text]
    _, _, ss = best_match(text)
    best_score, best_pair = 0, None
    for sp in range(3, len(words)-2):
        l, r = " ".join(words[:sp]), " ".join(words[sp:])
        _, li, ls = best_match(l); _, ri, rs = best_match(r)
        if li != ri and ls >= 60 and rs >= 60 and (ls+rs) > best_score:
            best_score, best_pair = ls+rs, (l, r)
    if best_pair and best_score > ss+30: return list(best_pair)
    return [text]

def best_match(text):
    best_s, best_i, best_t = -1, -1, ""
    for i, stim in enumerate(EIT_STIMULI):
        s = fuzzy(text, stim)
        if s > best_s: best_s, best_i, best_t = s, i, stim
    return best_t, best_i, best_s


def extract_responses(segs):
    """
    Dynamic alignment: fuzzy-match each segment to one of the 30 stimuli.
    Uses anchor gravity (bonus for expected position) to handle weak responses.
    """
    responses, expected = {}, 0
    for i, seg in enumerate(segs):
        if should_skip(seg["text"]):
            continue
        for part in split_merged(seg["text"]):
            # Score all 30 stimuli with position bonus
            best_i, best_s = -1, -1
            for j, stim in enumerate(EIT_STIMULI):
                bonus = 12 if j==expected else (8 if j==expected+1 else (4 if j==expected+2 else 0))
                s = fuzzy(part, stim) + bonus
                if s > best_s: best_s, best_i = s, j
            actual = fuzzy(part, EIT_STIMULI[best_i])

            if actual >= 40:
                target = best_i; expected = target + 1
            else:
                target = expected
                if expected > 0:
                    prev, old = expected-1, responses.get(expected, "")
                    if fuzzy(old+" "+part, EIT_STIMULI[prev]) > fuzzy(old, EIT_STIMULI[prev]):
                        target = prev; expected = prev + 1

            if target >= 30: continue
            sn, stim = target+1, EIT_STIMULI[target]

            if sn in responses:
                old = responses[sn]
                merged = old+" "+part
                if   fuzzy(merged, stim) > max(fuzzy(old,stim), actual) + 2: responses[sn] = merged
                elif actual > fuzzy(old, stim) + 3:                           responses[sn] = part
                elif actual >= 40 and actual >= fuzzy(old,stim)-5:            responses[sn] = part
            else:
                responses[sn] = part
            if target == expected and actual < 40: expected += 1
    return responses

print("✅ Alignment functions defined")

✅ Alignment functions defined


In [15]:
import openpyxl

def write_to_excel(excel_path, file_id, responses):
    """Write responses to column C. Missing sentences → '[no response]'."""
    wb = openpyxl.load_workbook(excel_path)
    sheet = next((wb[s] for s in wb.sheetnames if str(file_id) in s), None)
    if sheet is None:
        print(f"    ⚠️  No sheet found for ID '{file_id}'"); return 0
    filled = no_resp = 0
    for row in sheet.iter_rows(min_row=2):
        try:
            sn = int(row[0].value)
        except (ValueError, TypeError):
            continue
        if not 1 <= sn <= 30: continue
        if sn in responses:
            row[2].value = responses[sn]; filled  += 1
        else:
            row[2].value = "[no response]"; no_resp += 1
    wb.save(excel_path)
    print(f"    ✅ {filled} responses  +  {no_resp} × [no response]  →  sheet '{sheet.title}'")
    return filled + no_resp

print("✅ Excel writer defined")

✅ Excel writer defined


In [16]:
# ── Run Stage 2 on all transcripts ────────────────────────────────────────────
EXCEL_PATH = "./AutoEIT Sample Audio for Transcribing.xlsx"

output_folders = sorted(Path("./output").iterdir()) if Path("./output").exists() else []
tx_files = [(f.name, f/"transcript.txt") for f in output_folders
            if f.is_dir() and (f/"transcript.txt").exists()]

if not tx_files:
    print("❌ No transcript.txt files found. Run Stage 1 first.")
else:
    stage2_summary = []
    for folder_name, tx_path in tx_files:
        m   = re.match(r"(\d+)", folder_name)
        pid = str(int(m.group(1))) if m else folder_name
        print(f"\n{'═'*60}")
        print(f"  {folder_name}  (ID: {pid})")
        print(f"{'═'*60}")

        segs     = parse_transcript(str(tx_path))
        boundary = find_boundary(segs)
        spanish  = segs[boundary:]
        print(f"    {len(segs)} total segments  |  {len(spanish)} after boundary")

        responses = extract_responses(spanish)

        # Print results table
        print(f"\n  {'#':>3}  {'Stimulus (target)':^46}  Response")
        print(f"  {'─'*3}  {'─'*46}  {'─'*40}")
        for n in range(1, 31):
            stim = EIT_STIMULI[n-1][:44]
            resp = responses.get(n, "— [no response]")
            print(f"  {n:3d}  {stim:<46}  {resp}")

        if Path(EXCEL_PATH).exists():
            write_to_excel(EXCEL_PATH, pid, responses)
        else:
            print(f"    ⚠️  Excel not found at {EXCEL_PATH}")

        stage2_summary.append((folder_name, len(responses)))

    print(f"\n{'═'*60}")
    print("  STAGE 2 COMPLETE")
    print(f"{'═'*60}")
    for fname, n in stage2_summary:
        print(f"  ✅  {fname:<35}  {n}/30 responses")


════════════════════════════════════════════════════════════
  038010_EIT-2A  (ID: 38010)
════════════════════════════════════════════════════════════
    Boundary after seg #21  (158s: 'Now let's begin.')
    52 total segments  |  30 after boundary

    #                Stimulus (target)                 Response
  ───  ──────────────────────────────────────────────  ────────────────────────────────────────
    1  Quiero cortarme el pelo                         Quiero cortarme el pelo.
    2  El libro está en la mesa                        El libro está en la mesa.
    3  El carro lo tiene Pedro                         El carro no tiene pelo.
    4  Él se ducha cada mañana                         El se ducha cada mañana.
    5  ¿Qué dice usted que va a hacer hoy?             ¿Qué dice usted que va a hacer hoy?
    6  Dudo que sepa manejar muy bien                  Dudo que sepa manejar muy bien.
    7  Las calles de esta ciudad son muy anchas        Las calles de esta ciudad son muy a

---
## Section 6 — Download Outputs

Download the filled Excel (submission file) and the full transcript archive.

In [19]:
import shutil
from google.colab import files

# ── Rename Excel to submission filename ───────────────────────────────────────
src  = Path("./AutoEIT Sample Audio for Transcribing.xlsx")
dest = Path("./AutoEIT_Sample_Audio_for_Transcribing_FILLED.xlsx")
if src.exists():
    shutil.copy(src, dest)
    print(f"✅ Submission file ready: {dest.name}")
else:
    print("❌ Excel not found — check Stage 2 ran successfully")

# ── Zip the transcript outputs ────────────────────────────────────────────────
shutil.make_archive("output", "zip", "./output")
print(f"✅ Transcripts archived: output.zip")

# ── Download ──────────────────────────────────────────────────────────────────
print("\nDownloading files…")
files.download(str(dest))
files.download("output.zip")

✅ Submission file ready: AutoEIT_Sample_Audio_for_Transcribing_FILLED.xlsx
✅ Transcripts archived: output.zip



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile

with zipfile.ZipFile("output.zip", 'r') as zip_ref:
    zip_ref.extractall("./output")

In [ ]:
import json


# Remove problematic widget metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open("cleaned_notebook.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)